In [ ]:
"""
Conformer Generation Pipeline - THREADED VERSION (Fork-safe)
============================================================
"""

CONFIG = {
    "input":         "Mtb.xlsx",
    "sheet":         1,
    "name_col":      "Name",
    "smiles_col":    "SMILES",
    "outdir":        "conformers",

    "binding_site":  "Q-Loop",
    "ic50_col":      "IC50 μM",
    "ic50_max":      15,

    "energy_window": 5.0,
    "rmsd_cutoff":   0.7,
    "num_confs":     500,
    "max_iters":     200,
    "random_seed":   42,

    "force_field":      "MMFF94",
    "keep_hydrogens":   False,
    "max_conformers":   100,
    "verbose":          True,
    
    # Parallelization settings
    "n_workers":        4,        # Threads (safe for RDKit on all platforms)
    "use_threads":      True,     # Use threads instead of processes
}

import sys
import time
import logging
from pathlib import Path
from types import SimpleNamespace
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


# Thread-safe logger
class ThreadSafeLogger:
    def __init__(self, verbose=True):
        self.logger = logging.getLogger("confgen")
        self.lock = threading.Lock()
        
        if not self.logger.handlers:
            self.logger.setLevel(logging.DEBUG if verbose else logging.INFO)
            handler = logging.StreamHandler(sys.stdout)
            formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s','%H:%M:%S')
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
    
    def info(self, msg):
        with self.lock:
            self.logger.info(msg)
    
    def warning(self, msg):
        with self.lock:
            self.logger.warning(msg)
    
    def error(self, msg):
        with self.lock:
            self.logger.error(msg)
    
    def debug(self, msg):
        with self.lock:
            self.logger.debug(msg)


# ------------------------------------------------------------
# Energy + optimization
# ------------------------------------------------------------
def optimize_conformer(mol, conf_id, force_field, max_iters):
    try:
        if force_field == "MMFF94s":
            props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94s")
            if props:
                ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=conf_id)
                if ff:
                    ff.Minimize(maxIts=max_iters)
                    return True, ff.CalcEnergy()

        # fallback UFF
        AllChem.UFFOptimizeMolecule(mol, confId=conf_id, maxIters=max_iters)
        ff = AllChem.UFFGetMoleculeForceField(mol, confId=conf_id)
        if ff:
            return True, ff.CalcEnergy()

    except Exception:
        pass  # Silent fail in thread

    return False, None


# ------------------------------------------------------------
# RMSD clustering (energy-sorted greedy)
# ------------------------------------------------------------
def rmsd_clustering(mol, conf_ids, energies, rmsd_cutoff, max_keep):
    pairs = sorted(zip(conf_ids, energies), key=lambda x: x[1])

    selected = [pairs[0][0]]

    for conf_id, energy in pairs[1:]:
        keep = True
        for ref in selected:
            try:
                rmsd = rdMolAlign.GetBestRMS(mol, mol, conf_id, ref)
                if rmsd < rmsd_cutoff:
                    keep = False
                    break
            except Exception:
                continue

        if keep:
            selected.append(conf_id)
            if len(selected) >= max_keep:
                break

    return selected


# ------------------------------------------------------------
# SDF Writer (FIXED)
# ------------------------------------------------------------
def write_sdf(mol, outdir, name, logger=None):
    """Write molecule with all conformers to SDF file."""
    safe = "".join(c if c.isalnum() else "_" for c in name)
    path = outdir / f"{safe}.sdf"

    try:
        # Collect actual conformer objects — don't assume IDs are 0..N-1
        conformers = mol.GetConformers()
        num_confs = len(conformers)

        if num_confs == 0:
            if logger:
                logger.warning(f"{name}: No conformers to write")
            return None

        # Set molecule-level props once
        mol.SetProp("_Name", name)
        mol.SetProp("Total_Conformers", str(num_confs))

        writer = Chem.SDWriter(str(path))
        writer.SetKekulize(False)

        for conf in conformers:
            conf_id = conf.GetId()

            # Promote per-conformer props onto the molecule before writing
            if conf.HasProp("Energy"):
                mol.SetDoubleProp("Energy", conf.GetDoubleProp("Energy"))
                mol.SetProp("Energy_Unit", "RDKit_forcefield")
            if conf.HasProp("Rank"):
                mol.SetIntProp("Rank", conf.GetIntProp("Rank"))
            mol.SetProp("Conformer_Index", str(conf_id))

            # SDWriter.write(mol, confId=...) writes only that conformer — no copy needed
            writer.write(mol, confId=conf_id)

        writer.close()

        if logger:
            logger.debug(f"{name}: Wrote {num_confs} conformers to {path}")

        return path

    except Exception as e:
        if logger:
            logger.error(f"{name}: Failed to write SDF - {str(e)}")
        return None


# ------------------------------------------------------------
# Core function (thread-safe)
# ------------------------------------------------------------
def generate_conformers(smiles, name, cfg, logger, worker_id=None):
    """Generate conformers for a single molecule"""
    
    prefix = f"[W{worker_id}] " if worker_id is not None else ""
    
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            logger.warning(f"{prefix}{name}: Invalid SMILES")
            return None, name, False, "Invalid SMILES"

        mol = Chem.AddHs(mol)

        params = AllChem.ETKDGv3()
        params.randomSeed = cfg.random_seed if cfg.random_seed else (worker_id if worker_id else 42)
        params.numThreads = 1  # CRITICAL: Use single thread per molecule to avoid conflicts
        params.pruneRmsThresh = -1

        conf_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=cfg.num_confs, params=params))

        if not conf_ids:
            logger.warning(f"{prefix}{name}: No conformers embedded")
            return None, name, False, "No conformers embedded"

        # Optimize sequentially (thread-safe)
        valid_conf_ids = []
        energies = []

        for conf_id in conf_ids:
            success, energy = optimize_conformer(
                mol, conf_id, cfg.force_field, cfg.max_iters
            )

            if success and energy is not None:
                valid_conf_ids.append(conf_id)
                energies.append(energy)

        if not energies:
            logger.warning(f"{prefix}{name}: No successful optimizations")
            return None, name, False, "No successful optimizations"

        energies = np.array(energies)
        e_min = energies.min()

        # Energy filter
        mask = energies <= e_min + cfg.energy_window

        filtered_conf_ids = [cid for cid, m in zip(valid_conf_ids, mask) if m]
        filtered_energies = [e for e, m in zip(energies, mask) if m]

        if not filtered_conf_ids:
            logger.warning(f"{prefix}{name}: No conformers in energy window")
            return None, name, False, "No conformers in energy window"

        # RMSD clustering
        selected = rmsd_clustering(
            mol,
            filtered_conf_ids,
            filtered_energies,
            cfg.rmsd_cutoff,
            cfg.max_conformers
        )

        # Sort final by energy
        final_pairs = sorted(
            [(cid, energies[list(valid_conf_ids).index(cid)]) for cid in selected],
            key=lambda x: x[1]
        )

        # Build output molecule with selected conformers
        if not cfg.keep_hydrogens:
            mol_noH = Chem.RemoveHs(mol)
            out_mol = Chem.RWMol(mol_noH)
            out_mol.RemoveAllConformers()

            # Create atom index mapping (original -> no H)
            idx_map = {}
            j = 0
            for atom in mol.GetAtoms():
                if atom.GetAtomicNum() != 1:
                    idx_map[atom.GetIdx()] = j
                    j += 1

            for rank, (cid, energy) in enumerate(final_pairs):
                src_conf = mol.GetConformer(cid)
                new_conf = Chem.Conformer(out_mol.GetNumAtoms())

                for i_src, i_dst in idx_map.items():
                    pos = src_conf.GetAtomPosition(i_src)
                    new_conf.SetAtomPosition(i_dst, pos)

                new_conf.SetDoubleProp("Energy", float(energy))
                new_conf.SetIntProp("Rank", rank + 1)
                out_mol.AddConformer(new_conf, assignId=True)

            final_mol = out_mol.GetMol()

        else:
            out_mol = Chem.RWMol(mol)
            out_mol.RemoveAllConformers()

            for rank, (cid, energy) in enumerate(final_pairs):
                src_conf = mol.GetConformer(cid)
                new_conf = Chem.Conformer(out_mol.GetNumAtoms())

                for atom_idx in range(mol.GetNumAtoms()):
                    pos = src_conf.GetAtomPosition(atom_idx)
                    new_conf.SetAtomPosition(atom_idx, pos)

                new_conf.SetDoubleProp("Energy", float(energy))
                new_conf.SetIntProp("Rank", rank + 1)
                out_mol.AddConformer(new_conf, assignId=True)

            final_mol = out_mol.GetMol()

        # Set molecule properties
        final_mol.SetProp("_Name", name)
        final_mol.SetProp("SMILES", smiles)
        final_mol.SetIntProp("NumConformers", len(final_pairs))
        final_mol.SetProp("EnergyUnits", "RDKit_forcefield")

        # Write SDF file with all conformers
        sdf_path = write_sdf(final_mol, Path(cfg.outdir), name, logger)
        
        if sdf_path is None:
            logger.warning(f"{prefix}{name}: Failed to write SDF file")
            return None, name, False, "Failed to write SDF"

        logger.info(f"{prefix}{name}: ✓ Generated {len(final_pairs)} conformers -> {sdf_path.name}")
        return final_mol, name, True, "Success"

    except Exception as e:
        logger.error(f"{prefix}{name}: Exception - {str(e)}")
        return None, name, False, str(e)


# ------------------------------------------------------------
# I/O
# ------------------------------------------------------------
def load_data(cfg, logger):
    df = pd.read_excel(cfg.input, sheet_name=cfg.sheet)

    if cfg.binding_site and "Binding site" in df.columns:
        df = df[df["Binding site"] == cfg.binding_site]

    if cfg.ic50_col and cfg.ic50_col in df.columns:
        df = df[df[cfg.ic50_col] < cfg.ic50_max]

    df = df.dropna(subset=[cfg.smiles_col, cfg.name_col])

    return list(zip(df[cfg.name_col], df[cfg.smiles_col]))


# ------------------------------------------------------------
# Parallel processing wrapper (Thread-based for RDKit compatibility)
# ------------------------------------------------------------
def process_molecules_parallel(records, cfg, logger):
    """Process molecules in parallel using ThreadPoolExecutor (safe for RDKit)"""
    
    n_workers = cfg.n_workers if hasattr(cfg, 'n_workers') else 4
    
    logger.info(f"Starting parallel processing with {n_workers} workers (threads)")
    
    successful = 0
    failed = 0
    results = []
    
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        future_to_name = {}
        for idx, (name, smiles) in enumerate(records):
            worker_id = idx % n_workers
            future = executor.submit(generate_conformers, smiles, name, cfg, logger, worker_id)
            future_to_name[future] = name
        
        for future in as_completed(future_to_name):
            name = future_to_name[future]
            try:
                mol, mol_name, success, message = future.result(timeout=3600)
                
                if success and mol:
                    successful += 1
                else:
                    failed += 1
                    logger.warning(f"Failed: {name} - {message}")
                
                results.append((name, success))
                
            except Exception as e:
                failed += 1
                logger.error(f"Exception for {name}: {e}")
                results.append((name, False))
    
    logger.info(f"Parallel processing complete: {successful} succeeded, {failed} failed")
    return results


# ------------------------------------------------------------
# Sequential fallback (if parallel fails)
# ------------------------------------------------------------
def process_molecules_sequential(records, cfg, logger):
    """Fallback to sequential processing"""
    
    logger.info("Falling back to sequential processing...")
    
    successful = 0
    failed = 0
    
    for name, smiles in records:
        try:
            mol, _, success, message = generate_conformers(smiles, name, cfg, logger, 0)
            if success and mol:
                successful += 1
            else:
                failed += 1
                logger.warning(f"Failed: {name} - {message}")
        except Exception as e:
            failed += 1
            logger.error(f"Exception for {name}: {e}")
    
    return successful, failed


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():
    cfg = SimpleNamespace(**CONFIG)
    logger = ThreadSafeLogger(cfg.verbose)

    outdir = Path(cfg.outdir)
    outdir.mkdir(exist_ok=True)

    records = load_data(cfg, logger)
    logger.info(f"Loaded {len(records)} molecules")
    
    if not records:
        logger.warning("No molecules to process")
        return

    start_time = time.time()
    
    try:
        results = process_molecules_parallel(records, cfg, logger)
        successful = sum(1 for _, success in results if success)
        failed = len(results) - successful
    except Exception as e:
        logger.error(f"Parallel processing failed: {e}")
        successful, failed = process_molecules_sequential(records, cfg, logger)
    
    elapsed = time.time() - start_time
    
    logger.info("=" * 60)
    logger.info(f"FINAL SUMMARY:")
    logger.info(f"  Total molecules: {len(records)}")
    logger.info(f"  Successful: {successful}")
    logger.info(f"  Failed: {failed}")
    logger.info(f"  Time elapsed: {elapsed:.1f} seconds")
    if len(records) > 0:
        logger.info(f"  Average time per molecule: {elapsed/len(records):.1f} seconds")
    logger.info("=" * 60)


if __name__ == "__main__":
    main()

23:42:33 [INFO] Loaded 11 molecules
23:42:33 [INFO] Starting parallel processing with 4 workers (threads)
23:42:58 [DEBUG] CK-2-88: Wrote 4 conformers to conformers\CK_2_88.sdf
23:42:58 [INFO] [W3] CK-2-88: ✓ Generated 4 conformers -> CK_2_88.sdf
23:43:06 [DEBUG] MTD-403: Wrote 8 conformers to conformers\MTD_403.sdf
23:43:06 [INFO] [W2] MTD-403: ✓ Generated 8 conformers -> MTD_403.sdf
23:43:07 [DEBUG] CK-3-22 (1T): Wrote 32 conformers to conformers\CK_3_22__1T_.sdf
23:43:07 [INFO] [W1] CK-3-22 (1T): ✓ Generated 32 conformers -> CK_3_22__1T_.sdf
23:43:28 [DEBUG] LT-9: Wrote 4 conformers to conformers\LT_9.sdf
23:43:28 [INFO] [W2] LT-9: ✓ Generated 4 conformers -> LT_9.sdf
23:43:32 [DEBUG] CK-2-63: Wrote 16 conformers to conformers\CK_2_63.sdf
23:43:32 [INFO] [W0] CK-2-63: ✓ Generated 16 conformers -> CK_2_63.sdf
23:43:38 [DEBUG] PG-203: Wrote 16 conformers to conformers\PG_203.sdf
23:43:38 [INFO] [W1] PG-203: ✓ Generated 16 conformers -> PG_203.sdf
23:44:00 [DEBUG] SL-2-25: Wrote 12 con